# SCENARIO: “Corporate Research Assistant System”
# 🏢 Background Story
# A multinational company deploys an AI-powered business intelligence assistant.
# 👉 Employees can ask:
# - “What’s the latest news about Tesla?”
# - “What’s Tesla’s current stock price?”
# - “Give me a company profile instantly.”
# 👉 Instead of manually searching news sites, finance portals, and HR databases,
# 👉 AI fetches all the data in parallel, analyzes it, and generates a professional report.

# ⚙️ How it works (mapped to your pipeline):
# - Parallel Data Collection → AI gathers news, stock prices, and company profiles simultaneously.
# - LLM Analysis → AI interprets the combined data, highlighting key insights.
# - Report Generation → AI produces a polished, executive-ready report.

# 💡 Impact:
# - Saves analysts hours of manual research.
# - Provides real-time, consolidated insights for decision-making.
# - Empowers managers with instant reports for board meetings or investor updates.

In [ ]:
!pip install groq gradio nest_asyncio yfinance

import os
import asyncio
import nest_asyncio
import gradio as gr
import yfinance as yf
from groq import Groq

os.environ["GROQ_API_KEY"] = "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"

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

nest_asyncio.apply()

_original_asyncio_run = asyncio.run

def compatible_asyncio_run(main, *, debug=None, loop_factory=None):
    return _original_asyncio_run(main)

asyncio.run = compatible_asyncio_run


company_profiles = {
    "Tesla": {
        "industry": "Automotive and Clean Energy",
        "hq": "Austin, Texas, USA",
        "employees": "Approx. 140000",
        "summary": "Tesla designs and manufactures electric vehicles, battery energy storage systems, and solar products."
    },
    "Apple": {
        "industry": "Consumer Electronics and Software",
        "hq": "Cupertino, California, USA",
        "employees": "Approx. 160000",
        "summary": "Apple develops consumer electronics, software, and digital services including iPhone, Mac, and iCloud."
    },
    "Microsoft": {
        "industry": "Software and Cloud Computing",
        "hq": "Redmond, Washington, USA",
        "employees": "Approx. 220000",
        "summary": "Microsoft builds enterprise software, cloud platforms, productivity tools, and AI solutions."
    }
}


company_tickers = {
    "Tesla": "TSLA",
    "Apple": "AAPL",
    "Microsoft": "MSFT"
}


company_news = {
    "Tesla": [
        "Tesla expands EV production capacity and focuses on cost optimization.",
        "Tesla continues to push AI and autonomous driving investments.",
        "Analysts remain focused on delivery growth and margin trends."
    ],
    "Apple": [
        "Apple strengthens services revenue and ecosystem expansion.",
        "Apple continues work on AI features across devices and software.",
        "Analysts monitor iPhone demand and product refresh cycles."
    ],
    "Microsoft": [
        "Microsoft continues enterprise AI and cloud platform expansion.",
        "Azure growth remains a key focus area for market analysts.",
        "Microsoft deepens productivity and Copilot ecosystem integration."
    ]
}


async def web_search(company):
    await asyncio.sleep(1)

    if company in company_news:
        text = f"Latest News About {company}:\n"
        for i, news in enumerate(company_news[company], start=1):
            text += f"- {i}. {news}\n"
        return text.strip()

    return f"No news available for {company}."


async def get_stock_data(company):
    await asyncio.sleep(1)

    if company not in company_tickers:
        return f"Stock data not available for {company}."

    ticker_symbol = company_tickers[company]

    try:
        stock = yf.Ticker(ticker_symbol)
        info = stock.info

        current_price = info.get("currentPrice") or info.get("regularMarketPrice") or "Unavailable"
        market_cap = info.get("marketCap", "Unavailable")
        currency = info.get("currency", "")

        return (
            f"Stock Information:\n"
            f"- Company: {company}\n"
            f"- Ticker: {ticker_symbol}\n"
            f"- Current Price: {current_price} {currency}\n"
            f"- Market Cap: {market_cap}"
        )
    except Exception as e:
        return f"Could not fetch stock data for {company}. Error: {str(e)}"


async def fetch_company_profile(company):
    await asyncio.sleep(1)

    if company in company_profiles:
        profile = company_profiles[company]
        return (
            f"Company Profile:\n"
            f"- Company: {company}\n"
            f"- Industry: {profile['industry']}\n"
            f"- Headquarters: {profile['hq']}\n"
            f"- Employees: {profile['employees']}\n"
            f"- Summary: {profile['summary']}"
        )

    return f"No profile available for {company}."


async def parallel_research(company):
    results = await asyncio.gather(
        web_search(company),
        get_stock_data(company),
        fetch_company_profile(company),
        return_exceptions=True
    )

    news, stock, profile = results

    return {
        "news": news if not isinstance(news, Exception) else "News unavailable",
        "stock": stock if not isinstance(stock, Exception) else "Stock unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Profile unavailable"
    }


def decide_intent(user_query):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
You are a corporate research assistant intent classifier.

Classify the user's query into exactly one of these:
- news
- stock
- profile
- full_report

Rules:
- "latest news" -> news
- "stock price" or "current stock price" -> stock
- "company profile" -> profile
- "full report", "research report", "complete report" -> full_report

Return exactly one label.

User Query: {user_query}
"""
        }]
    )
    return response.choices[0].message.content.strip().lower()


def extract_company_name(user_query):
    for company in company_profiles.keys():
        if company.lower() in user_query.lower():
            return company
    return None


def analyse_text(text, company):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Analyze the following business intelligence data for {company} and provide:
1. Key corporate summary
2. Important market/business observations
3. Investor or management insight
4. Risks or opportunities
5. Simple executive-friendly explanation

Data:
{text}
"""
        }]
    )
    return response.choices[0].message.content


def generate_report(analysis, company):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""
Create a professional executive-ready report for {company}.

Use this analysis:
{analysis}

Keep it:
- clear
- structured
- concise but informative
- suitable for managers or board-level review
"""
        }]
    )
    return response.choices[0].message.content


async def full_pipeline(company, user_query):
    if not company:
        return "Please mention a supported company name like Tesla, Apple, or Microsoft."

    data = await parallel_research(company)
    intent = decide_intent(user_query)

    if intent == "news":
        combined_text = data["news"]
    elif intent == "stock":
        combined_text = data["stock"]
    elif intent == "profile":
        combined_text = data["profile"]
    else:
        combined_text = f"""
{data['news']}

{data['stock']}

{data['profile']}
""".strip()

    analysis = analyse_text(combined_text, company)
    report = generate_report(analysis, company)

    final_output = f"""
==============================
CORPORATE RESEARCH ASSISTANT OUTPUT
==============================

Company: {company}
Detected Intent: {intent}

RAW DATA:
{combined_text}

--------------------------------
AI ANALYSIS + EXECUTIVE REPORT:
--------------------------------
{report}
"""
    return final_output.strip()


def run_normal_mode():
    print("Corporate Research Assistant System")
    company_name = input("Enter Company Name (Tesla / Apple / Microsoft): ").strip()
    user_question = input("Ask your question: ").strip()

    if not company_name:
        company_name = extract_company_name(user_question)

    if not company_name or not user_question:
        print("Please enter both company name and question.")
        return

    result = asyncio.run(full_pipeline(company_name, user_question))
    print("\nFINAL OUTPUT:\n")
    print(result)


def corporate_assistant_ui(company_name, user_query):
    company_name = company_name.strip()
    user_query = user_query.strip()

    if not company_name:
        company_name = extract_company_name(user_query)

    if not company_name or not user_query:
        return "Please enter company name and question."

    return asyncio.run(full_pipeline(company_name, user_query))


run_normal_mode()


with gr.Blocks() as demo:
    gr.Markdown("# Corporate Research Assistant System")
    gr.Markdown("""
Supported companies:
- Tesla
- Apple
- Microsoft

Example questions:
- What's the latest news about Tesla?
- What's Tesla's current stock price?
- Give me a company profile instantly.
- Give me a full research report on Tesla.
""")

    company_input = gr.Textbox(
        label="Enter Company Name",
        placeholder="Example: Tesla"
    )

    query_input = gr.Textbox(
        label="Ask your question",
        placeholder="Example: Give me a full research report on Tesla."
    )

    output_box = gr.Textbox(
        label="Assistant Response",
        lines=24
    )

    submit_btn = gr.Button("Get Report")

    submit_btn.click(
        fn=corporate_assistant_ui,
        inputs=[company_input, query_input],
        outputs=output_box
    )

demo.launch(share=True, debug=True)

Corporate Research Assistant System

FINAL OUTPUT:

CORPORATE RESEARCH ASSISTANT OUTPUT

Company: Tesla
Detected Intent: news

RAW DATA:
Latest News About Tesla:
- 1. Tesla expands EV production capacity and focuses on cost optimization.
- 2. Tesla continues to push AI and autonomous driving investments.
- 3. Analysts remain focused on delivery growth and margin trends.

--------------------------------
AI ANALYSIS + EXECUTIVE REPORT:
--------------------------------
**Tesla Executive Report**

**Executive Summary:**
Tesla, a leading electric vehicle (EV) manufacturer, is expanding its production capacity and optimizing costs to maintain its market share in an increasingly competitive EV market. The company's strategic investments in artificial intelligence (AI) and autonomous driving technologies position it for long-term growth and innovation.

**Key Highlights:**

1. **Corporate Summary:** Tesla is focused on expanding production capacity and optimizing costs to drive growth and pro